# Experiment 2: sensitivity to one additional training seed

Experiment 2 adds 32 trials, not another full sweep: four bases × four dataset partitions × two tasks at training seed 43. Each repeats the predefined full-update reference (LR 3e-7, L2-SP 0.003, one_sample). Seed 42 is reused from experiment 1.

Partitions, monitor seed and evaluation seed stay fixed. Match dataset and recipe before differencing; never average the whole main grid into the reference. Two seeds show sensitivity at this recipe but do not estimate a reliable population variance or establish robustness across other knobs.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import display
from src.visualize import style
from src.visualize.figures import FigureSaver
from src.visualize import campaign as cp
from src.data.dataset_names import display_frame
from src.visualize.inputs import analysis_root
style.apply()
sink = FigureSaver('experiment2/01_seed_sensitivity')
report = cp.NotebookReport('Experiment 2: sensitivity to one additional training seed')


## 1. Reference and repetition coverage

The experiment-1 reference already belongs to its 512 trials. These 32 additional trials bring main plus seed-check training to 544 trials; experiment 3 remains separate.

In [ ]:
main = {track: cp.load_campaign(1,track) for track in ('pd','lgd')}
repeat = {track: cp.load_campaign(2,track) for track in ('pd','lgd')}
for run in repeat.values():
    cp.show(sink, cp.plot_coverage(run))
report.add('1. Repetition coverage', '\n\n'.join(cp.coverage_summary(c) for c in repeat.values()))

## 2. When do seed paths separate?

Only matched dataset/milestone observations appear. Gray paths are datasets and the colored path is their equally weighted mean difference; positive favors seed 43.

In [ ]:
pairs = {track: cp.seed_pairs(main[track],repeat[track]) for track in ('pd','lgd')}
for track in ('pd','lgd'):
    cp.show(sink, cp.plot_seed_trajectories(pairs[track], repeat[track].metric))
report.add('2. Seed trajectories', '\n\n'.join(track.upper()+'\n'+cp.effect_summary(pairs[track],'difference') for track in pairs))

## 3. Endpoint agreement by dataset

The diagonal means identical effects at both seeds. The paired-difference panel reveals cases where similar averages hide opposing dataset movements. Datasets are paginated rather than omitted.

In [ ]:
for track in ('pd','lgd'):
    cp.show(sink, cp.plot_seed_pairs(pairs[track],repeat[track].metric,endpoint=repeat[track].target))
report.add('3. Endpoint seed agreement', '\n\n'.join(track.upper()+'\n'+cp.effect_summary(pairs[track][pairs[track].updates.eq(repeat[track].target)] if not pairs[track].empty else pairs[track],'difference') for track in pairs))

## 4. Does final evaluation tell the same story?

Final benchmark effects are paired within each run against its own matched untuned control, then compared across seeds on shared datasets. Monitoring and benchmark values are never pooled.

In [ ]:
final_pairs = {track: cp.seed_pairs(main[track],repeat[track],benchmark=True) for track in ('pd','lgd')}
for track in ('pd','lgd'):
    cp.show(sink, cp.plot_seed_pairs(final_pairs[track],repeat[track].metric))
report.add('4. Benchmark seed agreement', '\n\n'.join(track.upper()+'\n'+cp.effect_summary(final_pairs[track],'difference') for track in final_pairs))

## 5. Cost and boundaries

Different training randomness can alter row exposure and numerical behavior. Report that beside score sensitivity, and keep conclusions confined to the repeated reference.

In [ ]:
for run in repeat.values():
    cp.show(sink, cp.plot_diagnostics(run))
report.add('5. Cost and limitations', 'One predefined recipe, two total seeds. Differences measure observed sensitivity; no reliable seed-distribution interval or grid-wide robustness claim follows.')

## Summary

The following text repeats the sections in order. It is included verbatim in `All_Results.md`.

In [ ]:
print(report.summary(sink))